# Barcode-by-sample molecule matrix

**Audience:** researchers moving from a `SummarizedExperiment` count assay to an explicit Python/SQLite molecule table.

**Learning goals:** construct a barcode × sample matrix from UMI representatives, distinguish molecule counts from supporting reads, visualize the matrix, and export an analysis-ready CSV.

## Outline

1. Load `molecule_counts`.
2. Pivot corrected barcodes across corrected sample indices.
3. Plot a compact heatmap.
4. Export the matrix and verify totals.

In [ ]:
from __future__ import annotations

import csv
import sqlite3
import subprocess
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
database = ROOT / 'examples' / 'output' / 'example.sqlite'
if not database.exists():
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'generate_synthetic_mapseq.py')], check=True)
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'run_example_pipeline.py')], check=True)
con = sqlite3.connect(database)

## Pivot UMI representatives into a matrix

`molecule_count` counts distinct UMI representatives. `read_count` reports sequencing/PCR support for those molecules. The matrix below uses molecules.

In [ ]:
rows = con.execute('SELECT barcode, sample_index, molecule_count, read_count FROM molecule_counts ORDER BY barcode, sample_index').fetchall()
known_indices = ['CGTGAT', 'ACATCG']
by_barcode = defaultdict(dict)
read_support = defaultdict(int)
for barcode, sample_index, molecules, reads in rows:
    if sample_index in known_indices:
        by_barcode[barcode][sample_index] = molecules
        read_support[barcode] += reads
barcodes = sorted(by_barcode, key=lambda barcode: (-read_support[barcode], barcode))
matrix = [[by_barcode[barcode].get(index, 0) for index in known_indices] for barcode in barcodes]
print('barcode (prefix)'.ljust(20), *[index.rjust(8) for index in known_indices])
for barcode, values in zip(barcodes, matrix):
    print((barcode[:16] + '…').ljust(20), *[str(value).rjust(8) for value in values])

## Visualize projection/sample presence

For the tiny synthetic example a heatmap is sufficient. Real studies should filter or order barcodes explicitly rather than plotting millions of rows.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, max(2.8, 0.6 * len(barcodes))))
image = ax.imshow(matrix, cmap='Blues', aspect='auto', vmin=0)
ax.set_xticks(range(len(known_indices)), known_indices)
ax.set_yticks(range(len(barcodes)), [barcode[:10] + '…' for barcode in barcodes])
ax.set(xlabel='Corrected RT sample index', ylabel='Barcode representative', title='Unique UMI representatives per barcode and sample')
for row_index, values in enumerate(matrix):
    for column_index, value in enumerate(values):
        ax.text(column_index, row_index, value, ha='center', va='center', color='black')
fig.colorbar(image, ax=ax, label='Molecule count')
plt.tight_layout()
plt.show()

In [ ]:
output = ROOT / 'notebooks' / 'output' / 'barcode_sample_molecules.csv'
output.parent.mkdir(parents=True, exist_ok=True)
with output.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.writer(handle)
    writer.writerow(['barcode', *known_indices])
    writer.writerows([[barcode, *values] for barcode, values in zip(barcodes, matrix)])
database_molecules = con.execute('SELECT sum(molecule_count) FROM molecule_counts WHERE sample_index IN (?, ?)', known_indices).fetchone()[0]
matrix_molecules = sum(sum(values) for values in matrix)
con.close()
assert matrix_molecules == database_molecules
print(f'Exported {len(barcodes)} barcodes and {matrix_molecules} molecules to {output}')

## Exercise and extensions

**Exercise:** define a qualitative presence matrix using `molecule_count >= 2`, then compare it with the quantitative matrix.

**Common pitfall:** read counts are not molecule counts. PCR-amplified reads from one UMI should not be treated as independent molecules.

**Extensions:** join sample names and anatomical metadata, normalize with spike-ins, and export a sparse matrix for large experiments.